In [ ]:
import pandas as pd
from tqdm import tqdm
import numpy as np

# -----------------------------------------------------------------------------
# 1. CONFIGURATION ET CHARGEMENT DES DONNÉES
# -----------------------------------------------------------------------------
ENTREPOT_PATH = '~/Bureau/utils/data/'
path_out = './'
df = {}

def import_df(df_name, path_data, sep=',', index_col=None):
    df[df_name] = pd.read_csv(
        path_data + df_name + '.csv', 
        sep=sep, 
        index_col=index_col, 
        low_memory=False
    ).replace({'\\r\\n': '\\n'}, regex=True)

def import_dfs(df_names, path_data, sep=',', index_col=None):
    for df_name in tqdm(df_names): 
        import_df(df_name, path_data, sep, index_col=index_col)

# Répartition des tables selon la présence/absence d'une colonne 'id' servant d'index primary
tables_with_id = [
    'sdc', 'domaine', 'dispositif', 'parcelle', 'zone', 'synthetise', 'reseau',
    'noeuds_realise', 'noeuds_synthetise', 'connection_synthetise', 'connection_synthetise_restructure',
    'plantation_perenne_phases_realise', 'plantation_perenne_phases_synthetise',
    'plantation_perenne_realise', 'plantation_perenne_synthetise',
    'action_realise', 'action_synthetise',
    'recolte_rendement_prix', 'composant_culture', 'espece', 'culture'
]

tables_without_id = [
    'liaison_reseaux', 'liaison_sdc_reseau',
    'itk_realise_agrege', 'itk_synthetise_agrege',
    'action_realise_agrege', 'action_synthetise_agrege',
    'noeuds_synthetise_restructure', 'typologie_can_culture',
    'poids_connexions_synthetise_rotation', 'recolte_rendement_prix_restructure'
]

print("Importation des tables avec ID...")
import_dfs(tables_with_id, ENTREPOT_PATH, sep=',', index_col='id')

print("Importation des tables sans ID...")
import_dfs(tables_without_id, ENTREPOT_PATH, sep=',')


Importation des tables avec ID...


100%|██████████| 20/20 [00:36<00:00,  1.84s/it]


Importation des tables sans ID...


100%|██████████| 10/10 [00:38<00:00,  3.90s/it]


In [42]:

# -----------------------------------------------------------------------------
# 2. SELECTION DES SDC (Stratégie de sélection robuste)
# -----------------------------------------------------------------------------
# On cherche les SDC qui possèdent à la fois du "Réalisé" (actions/récoltes/plantations)
# ET du "Synthétisé" (rotations/connexions) pour éviter d'avoir des tables vides.

sdc_with_harvest = df['recolte_rendement_prix']['sdc_id'].dropna().unique() if 'sdc_id' in df['recolte_rendement_prix'].columns else []
sdc_with_actions_real = df['action_realise_agrege']['sdc_id'].dropna().unique()
sdc_with_connections = df['itk_synthetise_agrege']['sdc_id'].dropna().unique()

# SDC qui combinent ces informations
candidate_sdcs = set(sdc_with_actions_real).intersection(set(sdc_with_connections))

if not candidate_sdcs:
    # Repli si l'intersection est vide
    candidate_sdcs = set(sdc_with_actions_real).union(set(sdc_with_connections))

studied_sdc_ids = list(candidate_sdcs)[:15]

print(f"Nombre de SdC retenus : {len(studied_sdc_ids)}")

# -----------------------------------------------------------------------------
# 3. FILTRAGE PAS À PAS (REMONTEES ET REDESCENTES)
# -----------------------------------------------------------------------------

# --- A. Entités Systèmes & Structure ---
df['sdc_test'] = df['sdc'].loc[df['sdc'].index.isin(studied_sdc_ids)]

df['dispositif_test'] = df['dispositif'].loc[
    df['dispositif'].index.isin(df['sdc_test']['dispositif_id'])
]

df['domaine_test'] = df['domaine'].loc[
    df['domaine'].index.isin(df['dispositif_test']['domaine_id'])
]

df['parcelle_test'] = df['parcelle'].loc[
    df['parcelle']['sdc_id'].isin(df['sdc_test'].index)
]

# --- B. Réseaux & Liaisons ---

df['liaison_sdc_reseau_test'] = df['liaison_sdc_reseau'].loc[
    df['liaison_sdc_reseau']['sdc_id'].isin(df['sdc_test'].index)
]
df['liaison_reseaux_test'] = df['liaison_reseaux'].loc[
    df['liaison_reseaux']['reseau_id'].isin(df['liaison_sdc_reseau_test']['reseau_id'])
]
df['reseau_test'] = df['reseau'].loc[
    (df['reseau'].index.isin(
        df['liaison_sdc_reseau_test']['reseau_id'].unique()
    )) |
    (df['reseau'].index.isin(
        df['liaison_reseaux_test']['reseau_parent_id'].unique()
    ))
]




# --- C. Volet Réalisé (ITK, Actions, Plantations, Nœuds) ---
df['itk_realise_agrege_test'] = df['itk_realise_agrege'].loc[
    df['itk_realise_agrege']['sdc_id'].isin(df['sdc_test'].index)
]

df['action_realise_agrege_test'] = df['action_realise_agrege'].loc[
    df['action_realise_agrege']['sdc_id'].isin(df['sdc_test'].index)
]

# Correction clés : action_realise s'appuie sur l'index de action_realise_agrege ou id_action
df['action_realise_test'] = df['action_realise'].loc[
    df['action_realise'].index.isin(df['action_realise_agrege_test'].index) |
    df['action_realise'].index.isin(df['action_realise_agrege_test']['action_id'] if 'action_id' in df['action_realise_agrege_test'] else [])
]

df['zone_test'] = df['zone'].loc[
    df['zone'].index.isin(df['itk_realise_agrege_test']['zone_id'])
]

df['noeuds_realise_test'] = df['noeuds_realise'].loc[
    df['noeuds_realise'].index.isin(df['itk_realise_agrege_test']['itk_id'])
]

# Plantation Pérenne Réalisé (Phase & Réalisé)
df['plantation_perenne_phases_realise_test'] = df['plantation_perenne_phases_realise'].loc[
    df['plantation_perenne_phases_realise'].index.isin(df['itk_realise_agrege_test']['itk_id'])
]

df['plantation_perenne_realise_test'] = df['plantation_perenne_realise'].loc[
    df['plantation_perenne_realise'].index.isin(df['itk_realise_agrege_test']['plantation_perenne_realise_id']) |
    df['plantation_perenne_realise'].index.isin(df['plantation_perenne_phases_realise_test']['plantation_perenne_realise_id'] if 'plantation_perenne_realise_id' in df['plantation_perenne_phases_realise_test'] else [])
]

# --- D. Volet Synthétisé (Rotations, Connexions, Nœuds) ---
df['synthetise_test'] = df['synthetise'].loc[
    df['synthetise']['sdc_id'].isin(df['sdc_test'].index)
]

df['itk_synthetise_agrege_test'] = df['itk_synthetise_agrege'].loc[
    df['itk_synthetise_agrege']['synthetise_id'].isin(df['synthetise_test'].index) |
    df['itk_synthetise_agrege']['sdc_id'].isin(df['sdc_test'].index)
]

df['action_synthetise_agrege_test'] = df['action_synthetise_agrege'].loc[
    df['action_synthetise_agrege']['sdc_id'].isin(df['sdc_test'].index)
]

df['action_synthetise_test'] = df['action_synthetise'].loc[
    df['action_synthetise'].index.isin(df['action_synthetise_agrege_test'].index) |
    df['action_synthetise'].index.isin(df['action_synthetise_agrege_test']['action_id'] if 'action_id' in df['action_synthetise_agrege_test'] else [])
]

# Connection synthétisée & restructurée
df['connection_synthetise_test'] = df['connection_synthetise'].loc[
    df['connection_synthetise'].index.isin(df['itk_synthetise_agrege_test']['itk_id'])
]

df['connection_synthetise_restructure_test'] = df['connection_synthetise_restructure'].loc[
    df['connection_synthetise_restructure']['id'].isin(df['connection_synthetise_test'].index) if 'id' in df['connection_synthetise_restructure'] else
    df['connection_synthetise_restructure'].index.isin(df['connection_synthetise_test'].index)
]

df['noeuds_synthetise_test'] = df['noeuds_synthetise'].loc[
    df['noeuds_synthetise'].index.isin(df['connection_synthetise_test']['cible_noeuds_synthetise_id'])
]

df['noeuds_synthetise_restructure_test'] = df['noeuds_synthetise_restructure'].loc[
    df['noeuds_synthetise_restructure']['id'].isin(df['noeuds_synthetise_test'].index) if 'id' in df['noeuds_synthetise_restructure'] else
    df['noeuds_synthetise_restructure'].index.isin(df['noeuds_synthetise_test'].index)
]

df['poids_connexions_synthetise_rotation_test'] = df['poids_connexions_synthetise_rotation'].loc[
    df['poids_connexions_synthetise_rotation']['connexion_id'].isin(df['connection_synthetise_test'].index)
]

df['plantation_perenne_phases_synthetise_test'] = df['plantation_perenne_phases_synthetise'].loc[
    df['plantation_perenne_phases_synthetise'].index.isin(df['itk_synthetise_agrege_test']['itk_id'])
]

df['plantation_perenne_synthetise_test'] = df['plantation_perenne_synthetise'].loc[
    df['plantation_perenne_synthetise'].index.isin(df['itk_synthetise_agrege_test']['plantation_perenne_synthetise_id'])
]


Nombre de SdC retenus : 15


In [43]:
df['action_realise_test'] = df['action_realise'].loc[
    df['action_realise'].index.isin(df['action_realise_agrege_test']['id'])
]
df['action_synthetise_test'] = df['action_synthetise'].loc[
    df['action_synthetise'].index.isin(df['action_synthetise_agrege_test']['id'])
]
df['culture_test'] = df['culture'].loc[
    df['culture'].index.isin(df['typologie_can_culture_test'].culture_id)
]
df['composant_culture_test'] = df['composant_culture'].loc[
    df['composant_culture']['culture_id'].isin(df['culture_test'].index)
]

df['recolte_rendement_prix_test'] = df['recolte_rendement_prix'].loc[
    df['recolte_rendement_prix']['action_id'].isin(df['action_realise_test'].index) |
    df['recolte_rendement_prix']['action_id'].isin(df['action_synthetise_test'].index)
]
df['recolte_rendement_prix_restructure_test'] = df['recolte_rendement_prix_restructure'].loc[
    df['recolte_rendement_prix_restructure']['id'].isin(df['recolte_rendement_prix_test'].index)
]

In [47]:
df['plantation_perenne_phases_synthetise_test']

,duree,type,plantation_perenne_synthetise_id
id,,,
fr.inra.agrosyst.api.entities.practiced.PracticedCropCyclePhase_177f58e8-eb71-4a8e-9d72-e032e515b375,0.0,PLEINE_PRODUCTION,fr.inra.agrosyst.api.entities.practiced.Practi...
fr.inra.agrosyst.api.entities.practiced.PracticedCropCyclePhase_84d09935-dd93-4518-863d-b84ca5f52b49,0.0,PLEINE_PRODUCTION,fr.inra.agrosyst.api.entities.practiced.Practi...
fr.inra.agrosyst.api.entities.practiced.PracticedCropCyclePhase_c7d39107-fc2b-4099-9951-d28b5cbe11e2,0.0,PLEINE_PRODUCTION,fr.inra.agrosyst.api.entities.practiced.Practi...
fr.inra.agrosyst.api.entities.practiced.PracticedCropCyclePhase_7a0ea239-f7bb-4881-980c-ff46e4f44d38,0.0,PLEINE_PRODUCTION,fr.inra.agrosyst.api.entities.practiced.Practi...
fr.inra.agrosyst.api.entities.practiced.PracticedCropCyclePhase_bbdcf103-fae9-4b39-8c4f-f30e43ea7a90,0.0,PLEINE_PRODUCTION,fr.inra.agrosyst.api.entities.practiced.Practi...
fr.inra.agrosyst.api.entities.practiced.PracticedCropCyclePhase_af8cccb8-4048-41ab-a71e-77f90cc8eccc,0.0,PLEINE_PRODUCTION,fr.inra.agrosyst.api.entities.practiced.Practi...
fr.inra.agrosyst.api.entities.practiced.PracticedCropCyclePhase_401b0ac8-6582-4594-b327-cc9e2eb8670c,0.0,PLEINE_PRODUCTION,fr.inra.agrosyst.api.entities.practiced.Practi...
fr.inra.agrosyst.api.entities.practiced.PracticedCropCyclePhase_081d27ff-626b-4e58-9e98-efe992c8c7f0,0.0,PLEINE_PRODUCTION,fr.inra.agrosyst.api.entities.practiced.Practi...
fr.inra.agrosyst.api.entities.practiced.PracticedCropCyclePhase_fa585940-357e-4522-b0d2-c3f63ba39898,0.0,PLEINE_PRODUCTION,fr.inra.agrosyst.api.entities.practiced.Practi...


In [44]:
df['espece_test'] = df['espece'].loc[
    df['espece'].index.isin(df['composant_culture_test']['espece_id'])
]

In [45]:
# --- F. Typologie ---
df['typologie_can_culture_test'] = df['typologie_can_culture'].loc[
    df['typologie_can_culture']['culture_id'].isin(df['noeuds_realise_test']['culture_id'] if 'culture_id' in df['noeuds_realise_test'] else []) |
    df['typologie_can_culture']['culture_id'].isin(df['plantation_perenne_realise_test']['culture_id'] if 'culture_id' in df['plantation_perenne_realise_test'] else [])
]

# -----------------------------------------------------------------------------
# 4. Bilan et Validation des volumes d'extraction
# -----------------------------------------------------------------------------
export_list = [
    'sdc', 'domaine', 'dispositif', 'parcelle', 'zone', 'synthetise', 'reseau',
    'liaison_reseaux', 'liaison_sdc_reseau', 'itk_realise_agrege', 'itk_synthetise_agrege',
    'noeuds_realise', 'noeuds_synthetise', 'noeuds_synthetise_restructure',
    'connection_synthetise', 'connection_synthetise_restructure',
    'poids_connexions_synthetise_rotation', 'plantation_perenne_phases_realise',
    'plantation_perenne_phases_synthetise', 'plantation_perenne_realise',
    'plantation_perenne_synthetise', 'action_realise', 'action_realise_agrege',
    'action_synthetise', 'action_synthetise_agrege', 'recolte_rendement_prix',
    'recolte_rendement_prix_restructure', 'composant_culture', 'espece',
    'typologie_can_culture'
]

print("\n--- Diagnostic du remplissage des tables ---")
for t in export_list:
    key = f"{t}_test"
    count = len(df[key]) if key in df else 0
    print(f"Table '{t}': {count} lignes")
    if key in df:
        df[key].to_csv(f"{path_out}{t}.csv")


--- Diagnostic du remplissage des tables ---
Table 'sdc': 15 lignes
Table 'domaine': 15 lignes
Table 'dispositif': 15 lignes
Table 'parcelle': 100 lignes
Table 'zone': 96 lignes
Table 'synthetise': 15 lignes
Table 'reseau': 28 lignes
Table 'liaison_reseaux': 19 lignes
Table 'liaison_sdc_reseau': 17 lignes
Table 'itk_realise_agrege': 98 lignes
Table 'itk_synthetise_agrege': 86 lignes
Table 'noeuds_realise': 91 lignes
Table 'noeuds_synthetise': 59 lignes
Table 'noeuds_synthetise_restructure': 59 lignes
Table 'connection_synthetise': 67 lignes
Table 'connection_synthetise_restructure': 6 lignes
Table 'poids_connexions_synthetise_rotation': 67 lignes
Table 'plantation_perenne_phases_realise': 7 lignes
Table 'plantation_perenne_phases_synthetise': 19 lignes
Table 'plantation_perenne_realise': 7 lignes
Table 'plantation_perenne_synthetise': 19 lignes
Table 'action_realise': 1047 lignes
Table 'action_realise_agrege': 1047 lignes
Table 'action_synthetise': 873 lignes
Table 'action_synthetise_